# 💻 GUÍA PRÁCTICA AVANZADA: ARQUITECTURA MODEL CONTEXT PROTOCOL (MCP)
## Versión Corregida para Databricks — Paquetes JSON-RPC en Vivo, Swapping de Modelos y Auto-Fallback Gemini → OpenRouter

Este notebook implementa la arquitectura MCP de manera robusta y estable para clústeres de Databricks:
- **Transporte**: Exclusivamente `stdio` (el único viable en la nube de Databricks; SSE requiere un servidor local en tu PC).
- **Modelos**: Google Gemini 2.5 Flash (SDK oficial) o modelos gratis vía OpenRouter (Nvidia Nemotron y Gemma 4 como respaldo).
- **Auto-fallback ante rate limit**: si Gemini agota su cuota (HTTP 429 / RESOURCE_EXHAUSTED), el orquestador conmuta de forma transparente a OpenRouter sin interrumpir la demo.
- **Visualización**: Tarjetas JSON-RPC con diseño neón que muestran cada mensaje del protocolo MCP en tiempo real.

---

## 📦 Paso 1: Instalación de Dependencias en el Clúster de Databricks
Instalamos todas las librerías directamente sobre el entorno del clúster y reiniciamos Python para que sean reconocidas de inmediato.

In [ ]:
%pip install mcp openai pandas scikit-learn numpy sqlalchemy pymysql python-dotenv google-generativeai

In [ ]:
# Reiniciar Python para que las nuevas librerías sean cargadas en memoria
%restart_python

---

## 🛠️ Paso 2: Importar Librerías
Importamos MCP, SDKs de IA, y utilidades de visualización. Usamos `sys.executable` para que Databricks levante el servidor MCP con el intérprete de Python correcto del clúster, sin depender de binarios externos como `uv`.

In [ ]:
import os
import sys
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
import google.generativeai as genai
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from IPython.display import display, HTML

# Utilidades del cliente MCP (limpieza de esquemas para Gemini y detección de
# errores de cuota). El boilerplate vive fuera del notebook para mantener
# limpia la presentación; revisa mcp_utils.py si quieres ver la implementación.
from mcp_utils import limpiar_schema_para_gemini, es_error_cuota

print(f"✅ Librerías cargadas. Python activo: {sys.executable}")

---

## 🔑 Paso 3: Carga y Verificación de Credenciales
Cargamos las variables privadas del archivo `.env` y verificamos el estado de cada clave disponible.

In [ ]:
load_dotenv(override=True)

gemini_key = os.getenv("GEMINI_API_KEY", "")
openrouter_key = os.getenv("OPENROUTER_API_KEY", "")

print("📋 Estado de Credenciales:")
print(f"   ➔ GEMINI_API_KEY    : {'✅ Configurada' if gemini_key else '❌ Vacía'}")
print(f"   ➔ OPENROUTER_API_KEY: {'✅ Configurada' if openrouter_key else '❌ Vacía'}")

---

## 🌐 Paso 4: Selección de Modelo de IA
Dos rutas de inferencia limpias, una por cada SDK:
- **[1] Google Gemini 2.5 Flash** — vía el SDK oficial `google-generativeai`.
- **[2] OpenRouter (modelos gratis)** — vía el SDK de OpenAI apuntado al endpoint de OpenRouter. Por defecto se enruta al modelo gratis `nvidia/nemotron-3-super-120b-a12b:free` y, si esa cuota se agota, el orquestador del Paso 9 hará auto-fallback al modelo gratis de respaldo `google/gemma-4-31b-it:free`.

In [ ]:
# ── Modelos de OpenRouter enrutados a la oferta gratis ──────────────────────
# Ambos slugs están verificados contra https://openrouter.ai/api/v1/models
# y comparten el sufijo `:free`, que es el mecanismo oficial de OpenRouter
# para enrutar la petición a la cuota gratuita del proveedor.
OPENROUTER_MODELO_PRIMARIO = "nvidia/nemotron-3-super-120b-a12b:free"
OPENROUTER_MODELO_FALLBACK = "google/gemma-4-31b-it:free"

print("=======================================================================")
print(" SELECCIÓN DE MODELO E INTEGRACIÓN:")
print("-----------------------------------------------------------------------")
print(" [1] -> Google Gemini 2.5 Flash       (SDK oficial google-generativeai)")
print(f" [2] -> OpenRouter Free Models        (Primario: {OPENROUTER_MODELO_PRIMARIO})")
print(f"                                       (Fallback: {OPENROUTER_MODELO_FALLBACK})")
print("=======================================================================\n")

opcion_modelo = input("🧠 Selecciona tu modelo (1-2, ENTER = [1] por defecto): ").strip()

if opcion_modelo == "2":
    ia_proveedor = "openrouter"
    modelo_activo = OPENROUTER_MODELO_PRIMARIO
else:
    ia_proveedor = "gemini"
    modelo_activo = "gemini-2.5-flash"

# Validaciones de credenciales
if ia_proveedor == "gemini" and not gemini_key:
    raise ValueError("❌ Elegiste Google Gemini pero GEMINI_API_KEY está vacía en tu .env.")
if ia_proveedor == "openrouter" and not openrouter_key:
    raise ValueError("❌ Elegiste OpenRouter pero OPENROUTER_API_KEY está vacía en tu .env.")
# Para que el auto-fallback de Gemini → OpenRouter funcione, ambas claves
# deben estar presentes. Si falta la de OpenRouter, avisamos pero no abortamos.
if ia_proveedor == "gemini" and not openrouter_key:
    print("⚠️ Aviso: OPENROUTER_API_KEY no configurada. Si Gemini agota su cuota,"
          " el auto-fallback no podrá activarse.")

print(f"\n✅ Proveedor: {ia_proveedor.upper()} | Modelo Activo: {modelo_activo}")

---

## ⚙️ Paso 5: Configurar Parámetros del Servidor MCP
Usamos `sys.executable` para garantizar que Databricks ejecute el Servidor MCP con el intérprete del clúster activo. Esto elimina definitivamente el error `FileNotFoundError: 'uv'`.

In [ ]:
server_params = StdioServerParameters(
    command=sys.executable,          # El intérprete Python activo del clúster
    args=["mcp_server.py"],          # El servidor MCP de nuestro proyecto
    env=os.environ.copy()            # Heredar las credenciales del .env
)

print(f"✅ Servidor MCP configurado con: {sys.executable} mcp_server.py")

---

## 🎨 Paso 6: Sniffer Visual de Paquetes JSON-RPC
Esta función dibuja tarjetas de diseño neón/glassmorphism que interceptan y muestran en tiempo real exactamente qué mensajes JSON-RPC viajan por el canal de comunicación del protocolo MCP:
- 🟦 **Azul**: Peticiones que envía el Cliente MCP.
- 🟩 **Verde**: Respuestas que retorna el Servidor MCP.
- 🟨 **Naranja**: Decisiones de acción dictadas por el modelo de IA.

In [ ]:
def mostrar_log_json_rpc(direccion, metodo, payload):
    colores = {
        "CLIENTE_ENVIANDO":    ("📤 CLIENTE MCP → SOLICITANDO ACCIÓN (OUTBOX)",  "#00c3ff", "rgba(0,195,255,0.05)"),
        "SERVIDOR_RESPONDIENDO": ("📥 SERVIDOR MCP → RETORNANDO RESPUESTA (INBOX)", "#00ff88", "rgba(0,255,136,0.05)"),
        "IA_DECIDIENDO":       ("🧠 INTELIGENCIA ARTIFICIAL → INVOCANDO HERRAMIENTA", "#ffaa00", "rgba(255,170,0,0.05)"),
    }
    titulo, borde, bg = colores.get(direccion, ("📋 LOG", "#888", "rgba(0,0,0,0.05)"))
    json_str = json.dumps(payload, indent=2, ensure_ascii=False)
    display(HTML(f"""
    <div style="border:2px solid {borde};background:{bg};padding:15px;border-radius:8px;
                margin:12px 0;font-family:Consolas,monospace;box-shadow:0 4px 15px rgba(0,0,0,.3);">
      <div style="font-weight:bold;color:{borde};margin-bottom:8px;font-size:1.1em;">{titulo}</div>
      <div style="font-size:.85em;color:#94a3b8;margin-bottom:6px;">Método: <code>{metodo}</code></div>
      <pre style="background:#0f172a;padding:10px;border-radius:5px;color:#38bdf8;
                  overflow-x:auto;border:1px solid #1e293b;margin:0;">{json_str}</pre>
    </div>"""))

print("✅ Sniffer visual JSON-RPC cargado.")

---

## 💬 Paso 7: Configuración Interactiva del Prompt
Presiona Enter para usar la instrucción predeterminada, o escribe tu propia consulta personalizada para el modelo de IA.

In [ ]:
prompt_predeterminado = (
    "Extrae 100 transacciones de la base de datos MySQL usando tus herramientas "
    "y luego transforma los datos para dejarlos listos para entrenar un modelo de ML."
)

print("=======================================================================")
print(" SUGERENCIAS DE INSTRUCCIONES PARA EL MODELO:")
print("-----------------------------------------------------------------------")
print(f" [Defecto]     -> {prompt_predeterminado}")
print(" [Sugerencia 1] -> 'Extrae solo 10 transacciones de MySQL sin transformar nada.'")
print(" [Sugerencia 2] -> 'Aplica la transformación de ML sobre el archivo temp_dataset.csv ya existente.'")
print("=======================================================================\n")

user_input = input("✍️ Instrucción (ENTER para usar la predeterminada): ").strip()
prompt_activo = user_input if user_input else prompt_predeterminado

print(f"\n✅ Prompt activo: '{prompt_activo}'")

---

## 🚀 Paso 8: Orquestador del Pipeline MCP con Visualización JSON-RPC y Auto-Fallback
Este es el núcleo del sistema. Conecta el Servidor MCP local, descubre las herramientas, consulta al LLM y ejecuta las acciones locales, todo mostrando el flujo de mensajes JSON-RPC con tarjetas de diseño premium.

Incluye dos mecanismos de resiliencia:
- **Backoff exponencial (2/4/8/16s)** ante errores transitorios 429/500 del LLM.
- **Auto-fallback Gemini → OpenRouter**: si Gemini agota su cuota incluso después de los reintentos, el código captura el error, registra un aviso visual y reinicia la orquestación contra el modelo gratis de respaldo de OpenRouter, sin pedir intervención manual.

In [ ]:
SYSTEM_PROMPT = (
    "Eres un agente científico de datos. Tu tarea es usar tus herramientas locales "
    "para extraer y transformar datos de MySQL. Si una herramienta retorna un error "
    "o aviso (como modo offline), analiza el mensaje y decide si continuar con el "
    "siguiente paso, reintentar con argumentos corregidos, o detenerte explicando "
    "el fallo. Responde de forma concisa en español."
)

# ── Constantes de reintentos ──────────────────────────────────────────────────
MAX_RETRIES = 4          # Máximo de reintentos por llamada a la API del LLM
BACKOFF_BASE_SECONDS = 2 # Base del backoff exponencial (2, 4, 8, 16 seg)
MAX_REACT_TURNS = 10     # Límite de seguridad para el bucle ReAct


class _CuotaAgotadaError(Exception):
    """Marcador interno que viaja desde `_send_with_retries` hasta el wrapper
    de auto-fallback cuando el LLM agotó su cuota tras todos los reintentos."""
    def __init__(self, original):
        super().__init__(str(original))
        self.original = original


async def _send_with_retries(send_coro_factory, descripcion="LLM"):
    """Ejecuta una corrutina de envío al LLM con reintentos y backoff exponencial
    para errores 429 (Rate Limit) y 500 (Server Error). Si tras agotar los
    reintentos el error sigue siendo de cuota, levanta `_CuotaAgotadaError`
    para que el wrapper externo pueda detonar el auto-fallback de proveedor.
    """
    for intento in range(1, MAX_RETRIES + 1):
        try:
            return await send_coro_factory()
        except Exception as e:
            error_str = str(e).lower()
            codigo = getattr(e, 'status_code', None) or getattr(e, 'code', None)
            es_rate_limit = (codigo == 429) or ('429' in error_str) or ('rate' in error_str and 'limit' in error_str) or es_error_cuota(e)
            es_server_error = (codigo == 500) or ('500' in error_str) or ('internal server error' in error_str)

            if (es_rate_limit or es_server_error) and intento < MAX_RETRIES:
                wait = BACKOFF_BASE_SECONDS ** intento
                etiqueta = '429 Rate Limit / Cuota' if es_rate_limit else '500 Server'
                print(f"⚠️ [{descripcion}] Error {etiqueta} "
                      f"en intento {intento}/{MAX_RETRIES}. Reintentando en {wait}s...")
                await asyncio.sleep(wait)
            else:
                if es_rate_limit:
                    raise _CuotaAgotadaError(e) from e
                raise


def _mostrar_aviso_fallback(modelo_origen, modelo_destino, motivo):
    """Tarjeta visual de aviso cuando se activa el auto-fallback de proveedor."""
    display(HTML(f"""
    <div style="border:2px solid #f97316;background:rgba(249,115,22,0.08);
                padding:18px;border-radius:10px;margin:12px 0;
                font-family:sans-serif;box-shadow:0 4px 18px rgba(249,115,22,.25);">
      <div style="font-weight:bold;color:#f97316;font-size:1.15em;margin-bottom:8px;">
        🛟 AUTO-FALLBACK ACTIVADO: cuota agotada en el proveedor primario
      </div>
      <div style="color:#fed7aa;font-size:0.95em;line-height:1.5;">
        <code style="color:#fff;">{modelo_origen}</code> agotó su cuota tras los reintentos.
        Conmutando de forma transparente a
        <code style="color:#fff;">{modelo_destino}</code> (OpenRouter free) para que
        la demo del protocolo MCP siga su curso sin intervención manual.
      </div>
      <div style="color:#94a3b8;font-size:0.8em;margin-top:8px;">Motivo: {motivo}</div>
    </div>
    """))


async def _ejecutar_pipeline(prompt, proveedor, modelo, session, tools_response):
    """Núcleo del bucle ReAct para un proveedor concreto. Recibe la sesión MCP
    ya inicializada para que el auto-fallback pueda reutilizarla en el mismo
    canal stdio sin reconectarse."""
    respuesta_final = None
    idx_call = 10
    print(f"\n🧠 Consultando al modelo '{modelo}' ({proveedor.upper()})...")

    if proveedor == "gemini":
        gemini_functions = [
            {
                "name": t.name,
                "description": t.description,
                "parameters": limpiar_schema_para_gemini(t.inputSchema),
            }
            for t in tools_response.tools
        ]
        genai.configure(api_key=gemini_key)
        model = genai.GenerativeModel(
            model_name=modelo,
            tools=gemini_functions,
            system_instruction=SYSTEM_PROMPT,
        )
        chat = model.start_chat(history=[])
        prompt_actual = prompt
        from google.generativeai import protos

        for turno in range(MAX_REACT_TURNS):
            response = await _send_with_retries(
                lambda p=prompt_actual: chat.send_message_async(p),
                descripcion=f"Gemini turno {turno+1}",
            )
            parts = response.candidates[0].content.parts
            tool_calls = [p.function_call for p in parts if p.function_call.name]

            if not tool_calls:
                respuesta_final = response.text
                break

            partes_respuesta = []
            for fn_call in tool_calls:
                nombre_fn = fn_call.name
                argumentos = dict(fn_call.args)

                mostrar_log_json_rpc(
                    "IA_DECIDIENDO", f"function_call → {nombre_fn}",
                    {"model": modelo, "function": nombre_fn, "arguments": argumentos},
                )
                mostrar_log_json_rpc(
                    "CLIENTE_ENVIANDO", "tools/call (REQUEST)",
                    {"jsonrpc": "2.0", "method": "tools/call",
                     "params": {"name": nombre_fn, "arguments": argumentos}, "id": idx_call},
                )

                resultado = await session.call_tool(nombre_fn, argumentos)
                resultado_texto = resultado.content[0].text

                mostrar_log_json_rpc(
                    "SERVIDOR_RESPONDIENDO", "tools/call (RESPONSE)",
                    {"jsonrpc": "2.0",
                     "result": {"content": [{"type": "text", "text": resultado_texto}]},
                     "id": idx_call},
                )

                partes_respuesta.append(
                    protos.Part(
                        function_response=protos.FunctionResponse(
                            name=nombre_fn,
                            response={"result": resultado_texto},
                        )
                    )
                )
                idx_call += 1

            prompt_actual = partes_respuesta
        else:
            respuesta_final = (
                f"[ADVERTENCIA] El bucle ReAct alcanzó el límite de {MAX_REACT_TURNS} turnos "
                f"sin que el modelo emitiera una respuesta final."
            )

    else:  # openrouter (SDK OpenAI)
        llm_client = AsyncOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=openrouter_key,
        )
        openrouter_tools = [
            {"type": "function", "function": {
                "name": t.name,
                "description": t.description,
                "parameters": t.inputSchema,
            }}
            for t in tools_response.tools
        ]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]

        for turno in range(MAX_REACT_TURNS):
            response = await _send_with_retries(
                lambda: llm_client.chat.completions.create(
                    model=modelo,
                    messages=messages,
                    tools=openrouter_tools,
                ),
                descripcion=f"OpenRouter turno {turno+1}",
            )
            mensaje = response.choices[0].message
            messages.append(mensaje)

            if not mensaje.tool_calls:
                respuesta_final = mensaje.content
                break

            for tool_call in mensaje.tool_calls:
                nombre_fn = tool_call.function.name
                argumentos = json.loads(tool_call.function.arguments)

                mostrar_log_json_rpc(
                    "IA_DECIDIENDO", f"tool_call → {nombre_fn}",
                    {"tool_call_id": tool_call.id, "model": modelo,
                     "function": nombre_fn, "arguments": argumentos},
                )
                mostrar_log_json_rpc(
                    "CLIENTE_ENVIANDO", "tools/call (REQUEST)",
                    {"jsonrpc": "2.0", "method": "tools/call",
                     "params": {"name": nombre_fn, "arguments": argumentos}, "id": idx_call},
                )

                resultado = await session.call_tool(nombre_fn, argumentos)
                resultado_texto = resultado.content[0].text

                mostrar_log_json_rpc(
                    "SERVIDOR_RESPONDIENDO", "tools/call (RESPONSE)",
                    {"jsonrpc": "2.0",
                     "result": {"content": [{"type": "text", "text": resultado_texto}]},
                     "id": idx_call},
                )

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": nombre_fn,
                    "content": resultado_texto,
                })
                idx_call += 1
        else:
            respuesta_final = (
                f"[ADVERTENCIA] El bucle ReAct alcanzó el límite de {MAX_REACT_TURNS} turnos "
                f"sin que el modelo emitiera una respuesta final."
            )

    return respuesta_final


async def correr_pipeline_mcp(prompt, proveedor, modelo):
    """Punto de entrada del pipeline MCP. Maneja la sesión stdio del servidor
    MCP y envuelve la ejecución del LLM en un try/except que detona el
    auto-fallback Gemini → OpenRouter ante cuota agotada (HTTP 429)."""

    print(f"🔌 [MCP] Conectando al Servidor local vía stdio...")
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mostrar_log_json_rpc(
                "CLIENTE_ENVIANDO", "tools/list",
                {"jsonrpc": "2.0", "method": "tools/list", "id": 1},
            )
            tools_response = await session.list_tools()
            mostrar_log_json_rpc(
                "SERVIDOR_RESPONDIENDO", "tools/list (RESPONSE)",
                {"jsonrpc": "2.0",
                 "result": {"tools": [
                     {"name": t.name, "description": t.description}
                     for t in tools_response.tools
                 ]},
                 "id": 1},
            )

            try:
                return await _ejecutar_pipeline(prompt, proveedor, modelo, session, tools_response)
            except _CuotaAgotadaError as cuota_err:
                # Solo conmuta automáticamente si veníamos por Gemini, hay clave
                # de OpenRouter, y el modelo de respaldo está definido.
                if proveedor != "gemini" or not openrouter_key:
                    raise cuota_err.original
                _mostrar_aviso_fallback(
                    modelo_origen=modelo,
                    modelo_destino=OPENROUTER_MODELO_FALLBACK,
                    motivo=str(cuota_err.original)[:200],
                )
                return await _ejecutar_pipeline(
                    prompt, "openrouter", OPENROUTER_MODELO_FALLBACK,
                    session, tools_response,
                )


print("✅ Orquestador cargado con auto-fallback Gemini → OpenRouter.")

---

## 🟢 Paso 9: Ejecutar el Pipeline en Vivo
Lanzamos la ejecución. Verás los paquetes JSON-RPC neón en tiempo real y al final la respuesta en lenguaje natural del modelo seleccionado.

In [ ]:
respuesta = await correr_pipeline_mcp(prompt_activo, ia_proveedor, modelo_activo)

---

## 📋 Paso 10: Respuesta Final del Modelo en Lenguaje Natural
Aquí se muestra el texto final que el modelo de IA devolvió al usuario como resultado de la orquestación completa.

In [ ]:
display(HTML(f"""
<div style="border:2px solid #a855f7;background:rgba(168,85,247,0.07);padding:20px;
            border-radius:10px;margin:12px 0;font-family:sans-serif;
            box-shadow:0 4px 20px rgba(168,85,247,0.2);">
  <div style="font-weight:bold;color:#a855f7;font-size:1.2em;margin-bottom:10px;">
    💬 RESPUESTA FINAL DEL MODELO ({modelo_activo.upper()})
  </div>
  <div style="color:#e2e8f0;font-size:1em;line-height:1.6;">
    {respuesta}
  </div>
</div>
"""))